# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NourhanFarag/nourhan-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Lane: Refresh / Content Opportunity Scoring

My primary task is a **ranking / scoring problem**.

The goal is not simply to classify pages as good or bad. The practical decision is to rank content pages so that an editor or SEO reviewer can inspect the most important candidates first.

Each page will eventually receive a review-priority score, and the pages with the highest scores will appear at the top of the review queue.

A classification model may later be used underneath the ranking to estimate the probability of a useful proxy outcome, but the final business output is still a ranked list of pages.

This matches the real decision because editorial capacity is limited: the team cannot review every page at once, so the useful question is "which pages should be reviewed first?"

In [1]:
from pathlib import Path
import pandas as pd

# Find the repository root whether the notebook runs
# from the repo root or from work/notebooks/
cwd = Path.cwd()

possible_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

repo_root = next(
    root
    for root in possible_roots
    if (root / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(f"Data source: {data_path}")
print(f"Rows: {len(df):,}")
print(f"Unique pages: {df['content_id'].nunique():,}")
print(f"Clients: {df['client_id'].nunique():,}")
print("ML task: Ranking / Scoring")

Data source: e:\FlyRank\nourhan-flyrank-ml-internship\data\raw\content_refresh_anonymized.csv
Rows: 30,000
Unique pages: 30,000
Clients: 32
ML task: Ranking / Scoring


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Provisional target / proxy

For this starter-data framing exercise, I will use an **observed decline indicator as a provisional proxy**.

The proxy will be:

- `1` when `trend_direction == "down"`
- `0` otherwise

This proxy represents whether the page was observed as declining in the current measurement window.

It is not the same as "this page definitely needs a refresh." A page may decline because of seasonality, changing search demand, competition, consolidation, measurement noise, or other factors.

The proxy is useful for learning the ML workflow with the starter dataset, but a stronger final capstone target would ideally use a future observed outcome, for example:

**features from a previous time window → decline or recovery in a later time window**

Because the proxy is derived from `trend_direction`, `trend_direction` and `trend_pct` must not be used as model features. Doing so would leak information about the answer into the model.

In [2]:
df["is_declining_proxy"] = (
    df["trend_direction"]
    .eq("down")
    .astype(int)
)

proxy_counts = (
    df["is_declining_proxy"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Not declining", 1: "Declining"})
)

print(proxy_counts)

decline_rate = df["is_declining_proxy"].mean()

print(
    f"\nObserved declining proxy rate: "
    f"{decline_rate:.1%}"
)

is_declining_proxy
Not declining    13738
Declining        16262
Name: count, dtype: int64

Observed declining proxy rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Primary success metric: Precision@50

My primary success metric will be **Precision@50**.

The final output is a ranked review queue, so the most important part of the ranking is the top of the list. If an editor can review only 50 pages, Precision@50 measures how many of those 50 pages match the chosen positive proxy outcome.

For example:

- Precision@50 = 0.60 means 30 of the top 50 pages match the positive proxy.
- Precision@50 = 0.80 means 40 of the top 50 pages match the positive proxy.

A model will not be considered useful simply because it has a high training score. It should outperform a transparent rule-based baseline on held-out data.

My provisional practical target is **Precision@50 of at least 0.60**, while also beating the baseline under the same validation design. This target may be revised later when the review capacity and final outcome definition become clearer.

In [3]:
k = 50
minimum_useful_pages = 30

provisional_precision_target = minimum_useful_pages / k

print(f"Review capacity (K): {k} pages")
print(
    f"Provisional target: at least "
    f"{minimum_useful_pages} useful pages in the top {k}"
)
print(
    f"Provisional Precision@{k} target: "
    f"{provisional_precision_target:.2f}"
)

print(
    f"Overall declining-proxy rate: "
    f"{df['is_declining_proxy'].mean():.2f}"
)

Review capacity (K): 50 pages
Provisional target: at least 30 useful pages in the top 50
Provisional Precision@50 target: 0.60
Overall declining-proxy rate: 0.54


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of analysis: one content page

The unit of analysis is **one pseudonymized content page**.

Each row represents one page and contains observable measurements about that page, such as search visibility, clicks, CTR, average position, content age, freshness, sessions, and engagement.

The ranking decision will therefore also be made at the page level:

**one row → one page → one review-priority score**

`content_id` and `client_id` are pseudonymized identifiers. They can be used for grouping, deduplication, and validation, but the identifier values themselves should not be used as predictive features.

In [4]:
lane_columns = [
    "content_id",
    "client_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "sessions_90d",
    "engagement_rate",
    "trend_direction",
    "is_declining_proxy",
]

lane_df = df[lane_columns].copy()

print(f"Lane dataframe shape: {lane_df.shape}")
print("One row = one pseudonymized content page")

lane_df.head(10)

Lane dataframe shape: (30000, 13)
One row = one pseudonymized content page


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,sessions_90d,engagement_rate,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20,17,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25,9,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,11,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22,78,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14,145,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,147,20,5,0.00,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,90,20,1,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,445,22,28,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,0.09,46.0,90,20,68,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.16,4.9,257,104,3,0.00,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why machine learning may add value beyond a fixed rule

A transparent fixed rule is an important baseline for this problem.

For example, a simple rule could prioritize pages that have high impressions, are stale, and show signs of decline.

However, review priority may depend on several signals interacting at the same time:

- search visibility;
- average position;
- CTR;
- content age;
- freshness;
- sessions;
- engagement;
- content depth;
- recent performance direction.

A page with low CTR alone may not deserve urgent review. Low CTR may become more meaningful when the page also has high impressions, a relatively strong search position, enough traffic volume, or signs of declining performance.

Writing every possible interaction as a manual if-statement could become complicated and brittle.

A machine-learning model may be able to learn combinations of signals and produce a continuous priority score that is more useful for ranking.

However, this is only a hypothesis at this stage. Machine learning must be compared with a transparent rule-based baseline under the same validation design. If the model does not improve the decision enough to justify its added complexity, the simpler rule should be preferred.

In [5]:
declining_with_demand = (
    df["trend_direction"].eq("down")
    & df["impressions_90d"].ge(100)
)

stale_visible = (
    df["days_since_last_update"].ge(180)
    & df["impressions_90d"].ge(500)
)

low_ctr_visible = (
    df["impressions_90d"].ge(500)
    & df["avg_position"].between(1, 20, inclusive="both")
    & df["ctr"].lt(0.5)
)

low_engagement_visible = (
    df["sessions_90d"].ge(30)
    & (
        (
            df["engagement_rate"].notna()
            & df["engagement_rate"].lt(30)
        )
        |
        (
            df["scroll_rate"].notna()
            & df["scroll_rate"].lt(30)
        )
    )
)

signal_count = (
    declining_with_demand.astype(int)
    + stale_visible.astype(int)
    + low_ctr_visible.astype(int)
    + low_engagement_visible.astype(int)
)

signal_summary = (
    signal_count
    .value_counts()
    .sort_index()
    .rename_axis("Number of review signals")
    .reset_index(name="Page count")
)

signal_summary["Percent of pages"] = (
    signal_summary["Page count"] / len(df) * 100
).round(1)

signal_summary

,Number of review signals,Page count,Percent of pages
0,0,11479,38.3
1,1,8923,29.7
2,2,7693,25.6
3,3,1902,6.3
4,4,3,0.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.